# Live Arb tick replay — Direct Match

Replays a `live_tick_logs/tsmc_ticks_YYYYMMDD.csv` recording (from the Live Arb
tab's Record button) and, after every tick, re-runs the **exact same**
Direct Match algorithm the live app uses — `logic/live_arb_logic.py::scan()`,
the same function `services/live_arb.py`'s background scan loop calls. We
import and call that function directly rather than re-implementing the
matching math, so this is guaranteed to agree with what the app would have
shown at that instant, not an approximation of it.

**How the replay works:** the tick log has one row per code per tick (not a
full-universe snapshot per tick). So we maintain an in-memory "current book"
per warrant/option code — exactly like `services/live_warrant.py`'s and
`services/live_options.py`'s own `_books` caches — and fold each tick into it
in timestamp order. After each fold we call `scan()` against the reconstructed
state, which is precisely the "read whatever's currently tracked" pattern
`services/live_arb.py::_scan_loop` uses in production.

**Known schema gap:** the CSV's `exercise_ratio` column (needed for every
warrant leg) was added after this feature first shipped. A recording made
before that fix will have blank `exercise_ratio` for every warrant row, and
`scan()` silently drops any warrant missing it — so an old file will show
zero (or very few) matches. The cell below checks for this and warns if it
finds it. If you hit this, re-record a fresh session and re-run.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Repo root on sys.path so `logic.live_arb_logic` imports exactly as it does
# for app.py/services/live_arb.py — this notebook lives in notebooks/, so the
# repo root is one directory up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from logic import live_arb_logic, iv_engine

print("arb kernel engine:", iv_engine.engine_info())

## Load the tick log

Point `CSV_PATH` at the file you downloaded (or leave it to auto-pick the
newest file under `live_tick_logs/`).

In [ ]:
# Set this explicitly to analyze a specific downloaded file, e.g.:
# CSV_PATH = Path("/path/to/tsmc_ticks_20260903.csv")
CSV_PATH = None

if CSV_PATH is None:
    candidates = sorted((REPO_ROOT / "live_tick_logs").glob("tsmc_ticks_*.csv"))
    if not candidates:
        raise FileNotFoundError("No tsmc_ticks_*.csv found under live_tick_logs/ — record a session first, "
                                 "or set CSV_PATH to a downloaded file.")
    CSV_PATH = candidates[-1]

print("Loading:", CSV_PATH)
raw = pd.read_csv(CSV_PATH, parse_dates=["ts"])
raw["expiry"] = pd.to_datetime(raw["expiry"], errors="coerce").dt.date
raw = raw.sort_values("ts", kind="stable").reset_index(drop=True)
print(f"{len(raw):,} ticks — {raw['kind'].value_counts().to_dict()}")

missing_ratio = raw[(raw["kind"] == "warrant") & raw["exercise_ratio"].isna()]
if len(missing_ratio):
    pct = len(missing_ratio) / (raw["kind"] == "warrant").sum() * 100
    print(f"WARNING: {len(missing_ratio):,} warrant ticks ({pct:.0f}%) have no exercise_ratio "
          f"— these will be silently excluded from every match below. "
          f"See the schema-gap note above.")

raw.head()

## Replay: fold each tick into a live book cache, then re-run `scan()`

`warrant_books`/`option_books` mirror `services/live_warrant.py::_books` /
`services/live_options.py::_books` — each tick fully replaces that one
code's cached row (both bid and ask sides), exactly like the app's own
`_handle_message`. After folding a tick we pass everything tracked so far to
`live_arb_logic.scan(warrant_rows, option_rows, today)` — the same call
`services/live_arb.py`'s scan loop makes on every iteration, with the same
default `max_dte_diff` — and record whatever pairs come back, tagged with the
triggering tick.

This calls `scan()` once per tick (not once per poll), so it is a strictly
finer-grained replay than what the UI ever actually showed you live — the UI
only samples the scan loop's state every 500ms. Runtime is dominated by the
number of ticks × one `scan()` call each (`direct_pairs` itself measures
~8ms for TSMC's full universe per `logic/live_arb_logic.py`'s docstring); set
`STRIDE > 1` below to only rescan every Nth tick if you want a quicker,
coarser look first.

In [ ]:
def _clean(v):
    """NaN/NaT -> None; everything else passed through unchanged. build_direct_arrays
    treats a present-but-falsy value differently from a missing one (e.g. dte<=0 is a
    real drop reason, a missing maturity is a different one), so this must produce a
    real None, never a NaN that happens to be truthy."""
    if v is None or v is pd.NaT:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    return v


def _fold_warrant_tick(row):
    return {
        "code": row.code,
        "name": _clean(row.name) or row.code,
        "type": _clean(row.type),
        "strike": _clean(row.strike),
        "exercise_ratio": _clean(row.exercise_ratio),
        "maturity": _clean(row.expiry),
        "best": {
            "bid": _clean(row.bid), "ask": _clean(row.ask),
            "bid_size": _clean(row.bid_size), "ask_size": _clean(row.ask_size),
        },
    }


def _fold_option_tick(row):
    return {
        "code": row.code,
        "name": _clean(row.name) or row.code,
        "type": _clean(row.type),
        "strike": _clean(row.strike),
        "expiry": _clean(row.expiry),
        "best": {
            "bid": _clean(row.bid), "ask": _clean(row.ask),
            "bid_size": _clean(row.bid_size), "ask_size": _clean(row.ask_size),
        },
    }


def replay_direct_match(df, stride=1, progress_every=5000):
    warrant_books, option_books = {}, {}
    out = []
    n = len(df)
    for i, row in enumerate(df.itertuples(index=False)):
        if row.kind == "warrant":
            warrant_books[row.code] = _fold_warrant_tick(row)
        else:
            option_books[row.code] = _fold_option_tick(row)

        if stride > 1 and (i + 1) % stride != 0:
            continue

        hits = live_arb_logic.scan(list(warrant_books.values()), list(option_books.values()), row.ts.date())
        for h in hits:
            out.append({"tick_ts": row.ts, "tick_kind": row.kind, "tick_code": row.code, **h})

        if progress_every and (i + 1) % progress_every == 0:
            print(f"{i + 1:,}/{n:,} ticks replayed, {len(out):,} hit-rows so far")

    return pd.DataFrame(out)


STRIDE = 1  # increase to e.g. 10 for a faster, coarser first pass
hits_df = replay_direct_match(raw, stride=STRIDE)
print(f"\n{len(hits_df):,} (tick, pair) hit-rows across {raw['ts'].nunique():,} distinct tick timestamps")
hits_df.head()

## Quick summary

`hits_df` has one row per (tick, matched warrant/option pair) — exactly
`scan()`'s own output fields (`price_diff`, `price_diff_pct`, `riskless`,
strikes, DTEs, ...) plus which tick triggered it. A tick with no active pair
contributes no rows, so gaps in `tick_ts` are "nothing was arbable then,"
not missing data.

In [ ]:
if len(hits_df):
    pair_counts = (hits_df.groupby(["warrant_code", "option_code"])
                   .agg(n_ticks=("tick_ts", "count"),
                        max_price_diff=("price_diff", "max"),
                        max_price_diff_pct=("price_diff_pct", "max"),
                        first_seen=("tick_ts", "min"),
                        last_seen=("tick_ts", "max"))
                   .sort_values("n_ticks", ascending=False))
    display(pair_counts.head(20))

    per_tick_count = hits_df.groupby("tick_ts").size()
    print(f"\nPairs active on {len(per_tick_count):,} distinct ticks "
          f"(max {per_tick_count.max()} simultaneous pairs, "
          f"riskless on {hits_df['riskless'].sum():,}/{len(hits_df):,} hit-rows)")
else:
    print("No active pairs found across the whole replay — check the exercise_ratio "
          "warning above if this file predates that CSV column.")

## Save

Writes the full per-tick hit table next to the source CSV for further
analysis outside this notebook.

In [ ]:
out_path = CSV_PATH.with_name(CSV_PATH.stem + "_direct_match_hits.csv")
hits_df.to_csv(out_path, index=False)
print("Saved:", out_path)